# 20 · 执行环境与工作树：先证明动作发生在哪里

命令正确且退出 0，仍可能检查了错误 checkout。环境身份回答“谁执行”，cwd 回答“在哪里”，owner 记录回答“归哪个聊天管理”；三者不能互相替代。
本章只使用一个新建临时 Git 仓库，在其中创建两个 detached worktree，真实运行进程与归属竞争。
不会对本教材仓库、用户项目或固定 Codex 源码执行 Git 写操作，也不模拟远端环境已通过验收。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [E20-10 · default_environment_ids/get_environment/remove_environment](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/exec-server/src/environment.rs#L386-L428)：默认环境 ID 排第一，其余源自 HashMap；get 克隆句柄；remove 不改默认 ID也不使旧句柄失效。
- [E20-20 · owner/bind_thread](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/worktree/src/metadata.rs#L19-L90)：owner 记录是 version=1、ownerThreadId；临时完整 JSON 通过 persist_noclobber 原子发布，同 owner 幂等，竞争 owner 拒绝。
- [E20-21 · WorktreeManager::remove](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/worktree/src/lib.rs#L284-L328)：remove 先从本仓库受管清单确认对象，拒删当前 checkout，额外拒绝 ignored 文件，再用无 force 的 git worktree remove。


## 实验 1 · 从同一个提交创建两份真实 checkout


工作树名相同不代表路径或身份相同。先自建最小 fixture 仓库并提交，再从同一 HEAD 分出两个 worktree。
Git helper 清除继承的仓库重定向变量，禁用用户配置与 hooks，避免把实验指向外部仓库。只需要本机 git 与 Python。


In [2]:
wt = load_example('20', 'ownership_probe')
run_dir = fresh_run_dir('ch20')
repo = run_dir/'fixture-repo'
repo.mkdir()
wt.git(repo,'init','-q')
(repo/'task_list.py').write_text('def remaining(tasks):\n    return [t for t in tasks if not t["done"]]\n')
wt.git(repo,'add','task_list.py')
wt.git(repo,'-c','user.name=Notebook Fixture','-c','user.email=fixture@example.invalid',
       '-c','commit.gpgsign=false','commit','-qm','disposable baseline')
revision = wt.git(repo,'rev-parse','HEAD').stdout.strip()
a, b = run_dir/'a'/'task-list', run_dir/'b'/'task-list'
for checkout in (a,b):
    checkout.parent.mkdir()
    wt.git(repo,'worktree','add','--detach',str(checkout),revision)
assert a != b
assert all(wt.git(p,'rev-parse','HEAD').stdout.strip()==revision for p in (a,b))
assert wt.owner(a) is None and wt.owner(b) is None
print({'fixture_revision':revision, 'checkouts':[str(p.relative_to(run_dir)) for p in (a,b)]})


{'fixture_revision': '88be20d3172d37a33cf035746b46f00c18c1637b', 'checkouts': ['a/task-list', 'b/task-list']}


## 实验 2 · owner 是独立的原子发布记录


创建 checkout 不自动取得 owner。发布完整 JSON 后，重复同 owner 是幂等，其他 owner 必须拒绝。
教学实现用同文件系统硬链接做 no-clobber 发布；这里只验证 POSIX 本机机制，不是跨平台通用 worktree 管理器。


In [3]:
assert wt.bind(a,'thread-A') == 'thread-A'
before = wt.owner_path(a).read_bytes()
assert wt.bind(a,'thread-A') == 'thread-A'
assert wt.owner_path(a).read_bytes() == before
with expect.assertRaisesRegex(ValueError,'different owner'):
    wt.bind(a,'thread-other')
assert wt.owner(a) == 'thread-A'
assert wt.owner_path(a) != wt.owner_path(b)
print({'owner_a':wt.owner(a),'owner_b':wt.owner(b),'metadata_separate':True})


{'owner_a': 'thread-A', 'owner_b': None, 'metadata_separate': True}


## 实验 3 · 两个 owner 竞争，恰有一个获胜


普通“先判断文件不存在、再写入”存在竞态。Barrier 让两个真实线程同时竞争 b 的 owner，
原子 no-clobber 确保不会互相覆盖。胜者不固定，断言只检查一胜一败与最终记录一致。


In [4]:
from concurrent.futures import ThreadPoolExecutor
import threading
barrier = threading.Barrier(2)
def contender(owner):
    barrier.wait(timeout=3)
    try:
        return {'owner':owner,'status':'won','value':wt.bind(b,owner)}
    except ValueError as error:
        return {'owner':owner,'status':'rejected','reason':str(error)}
with ThreadPoolExecutor(max_workers=2) as pool:
    outcomes = list(pool.map(contender,['thread-B','thread-C']))
winners = [x for x in outcomes if x['status']=='won']
assert len(winners)==1 and wt.owner(b)==winners[0]['owner']
assert sum(x['status']=='rejected' for x in outcomes)==1
print({'race':outcomes,'persisted_owner':wt.owner(b)})


{'race': [{'owner': 'thread-B', 'status': 'rejected', 'reason': 'different owner'}, {'owner': 'thread-C', 'status': 'won', 'value': 'thread-C'}], 'persisted_owner': 'thread-C'}


## 实验 4 · 进程必须在目标 checkout 验证 cwd


两份 checkout 各运行真实 Python 子进程并写不同 marker。进程同时报告实际 cwd，
把“我打算在 A 跑”变成“进程实际在 A 跑”的可核对证据。这仍是两个本地目录，不是两个隔离 OS 环境。


In [5]:
import subprocess, sys, json
program = 'from pathlib import Path; import sys,json; Path("marker.txt").write_text(sys.argv[1]); print(json.dumps({"cwd":str(Path.cwd()),"marker":Path("marker.txt").read_text()}))'
observed = []
for checkout, marker in ((a,'A'),(b,'B')):
    process = subprocess.run([sys.executable,'-c',program,marker],cwd=checkout,capture_output=True,text=True,check=True)
    result = json.loads(process.stdout)
    assert Path(result['cwd']).resolve()==checkout.resolve()
    assert result['marker']==marker
    observed.append(result)
assert (a/'marker.txt').read_text() != (b/'marker.txt').read_text()
assert not (repo/'marker.txt').exists()
print(observed)


[{'cwd': '<project>/runs/notebooks/ch20-20261113T124122Z-d621272e/a/task-list', 'marker': 'A'}, {'cwd': '<project>/runs/notebooks/ch20-20261113T124122Z-d621272e/b/task-list', 'marker': 'B'}]


## 实验 5 · 脏目录不可强删，恢复计划不等于恢复提交


真实 git worktree remove 应拒绝含未跟踪文件的目录。随后让目标文件出现用户改动，restore_plan 必须报 conflict 且不写文件。
即使当前内容匹配，只能得到 ready_for_commit_recheck：实际写回前还要再次检查，才能关闭检查与写入之间的竞态。
本章保留新建实验目录供核查；没有实现 archive/restore，也不执行任何强制清理。


In [6]:
refused = wt.git(repo,'worktree','remove',str(a),check=False)
assert refused.returncode != 0 and (a/'marker.txt').exists()
target = a/'task_list.py'
snapshot = target.read_bytes()
target.write_bytes(snapshot+b'# concurrent synthetic user edit\n')
current = target.read_bytes()
conflict = wt.restore_plan(snapshot,snapshot,current)
assert conflict['status']=='conflict' and target.read_bytes()==current
ready = wt.restore_plan(snapshot,current,current)
assert ready['status']=='ready_for_commit_recheck'
assert target.read_bytes()==current
assert wt.git(repo,'status','--porcelain').stdout == ''
print({'dirty_remove_exit':refused.returncode,'stderr':refused.stderr.strip(),
       'restore':conflict['status'],'matching_case':ready['status'],'source_checkout_clean':True})


{'dirty_remove_exit': 128, 'stderr': "fatal: '<project>/runs/notebooks/ch20-20261113T124122Z-d621272e/a/task-list' contains modified or untracked files, use --force to delete it", 'restore': 'conflict', 'matching_case': 'ready_for_commit_recheck', 'source_checkout_clean': True}


## 你现在可以解释什么

能区分 checkout、环境、owner 和活动状态，知道为何取消/失败后不能按路径猜测副作用归属。这里没有远端 executor、权限物化、OS 隔离、真实环境撤销或可提交的文件恢复实现。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
